# pipelines merge

In [ ]:
#%matplotlib inline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy import stats
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
Data_all = pd.read_excel("E:/jupyter lab/可压性项目产能预测-工作站结果/Data_all_forblindtest.xlsx")
feature_name = ['DEPTH','DT','ZDEN','GR','CNCF','M2RX','POR','PERM','SW']
X = Data_all[feature_name].values
y = Data_all['PROD'].values
y[y<0] = 0
X[X[:,6]<0,6] = 0
X[X[:,7]<0,7] = 0
X[X[:,8]>100,8] = 100
well_ind = Data_all['well_ind'].values
DEPTH = Data_all['DEPTH'].values

In [ ]:
# 划分训练集和测试集
X_train, X_test, y_train, y_test, well_ind_train, well_ind_test, DEPTH_train, DEPTH_test = train_test_split(X, y, well_ind, DEPTH, test_size=0.2, random_state=42)

# 定义 XGBoost，随机森林以及神经网络的 pipeline
pipelines = [
    ('ps-xgb', Pipeline([('scl', StandardScaler()),
                         ('pca', PCA()),
                         ('clf', XGBRegressor(random_state=42))])),
    ('ps-rf', Pipeline([('scl', StandardScaler()),
                        ('pca', PCA()),
                        ('clf', RandomForestRegressor(random_state=42))])),
    ('ps-nn', Pipeline([('scl', StandardScaler()),
                        ('pca', PCA()),
                        ('clf', MLPRegressor(random_state=42))])),
    ('pfs-xgb', Pipeline([('scl', StandardScaler()),
                          ('poly', PolynomialFeatures()),
                          ('clf', XGBRegressor(random_state=42))])),
    ('pfs-rf', Pipeline([('scl', StandardScaler()),
                         ('poly', PolynomialFeatures()),
                         ('clf', RandomForestRegressor(random_state=42))])),
    ('pfs-nn', Pipeline([('scl', StandardScaler()),
                         ('poly', PolynomialFeatures()),
                         ('clf', MLPRegressor(random_state=42))])),
    ('pr-xgb', Pipeline([('scl', RobustScaler()),
                         ('pca', PCA()),
                         ('clf', XGBRegressor(random_state=42))])),
    ('pr-rf', Pipeline([('scl', RobustScaler()),
                        ('pca', PCA()),
                        ('clf', RandomForestRegressor(random_state=42))])),
    ('pr-nn', Pipeline([('scl', RobustScaler()),
                        ('pca', PCA()),
                        ('clf', MLPRegressor(random_state=42))]))
]

# 设定每个模型的参数网格，包括主成分数量
param_grids = [
    {'pca__n_components': [2, 4, 6],  # 主成分数量
     'clf__learning_rate': [0.01, 0.1, 0.5],
     'clf__n_estimators': [50, 100, 200],
     'clf__max_depth': [1, 2, 3],
     'clf__min_child_weight': [1, 3, 5],
     'clf__booster': ['gbtree', 'gblinear']},  # XGBoost
    {'pca__n_components': [2, 4, 6],  # 主成分数量
     'clf__n_estimators': [400, 500, 600],
     'clf__max_depth': [2, 6, 10],
     'clf__min_samples_split': [10, 15, 20],
     'clf__min_samples_leaf': [4, 6, 8],
     'clf__max_features': ['auto', 'sqrt'],
     'clf__bootstrap': [True, False]},  # Random Forest
    {'pca__n_components': [2, 4, 6],  # 主成分数量
     'clf__hidden_layer_sizes': [(50,), (100,), (200,), (100, 50), (200, 100), (300, 200, 100), (400, 300, 200, 100)],
     'clf__alpha': [0.0001, 0.001, 0.01, 0.1],
     'clf__max_iter': [3000, 5000, 7000]},  # Neural Network
    {'poly__degree': [2, 3, 4],  # 多项式的阶数
     'clf__learning_rate': [0.01, 0.1, 0.5],
     'clf__n_estimators': [50, 100, 200],
     'clf__max_depth': [1, 2, 3],
     'clf__min_child_weight': [1, 3, 5],
     'clf__booster': ['gbtree', 'gblinear']},  # XGBoost
    {'poly__degree': [2, 3, 4],  # 多项式的阶数
     'clf__n_estimators': [400, 500, 600],
     'clf__max_depth': [2, 6, 10],
     'clf__min_samples_split': [10, 15, 20],
     'clf__min_samples_leaf': [4, 6, 8],
     'clf__max_features': ['auto', 'sqrt'],
     'clf__bootstrap': [True, False]},  # Random Forest
    {'poly__degree': [2, 3, 4],  # 多项式的阶数
     'clf__hidden_layer_sizes': [(50,), (100,), (200,), (100, 50), (200, 100), (300, 200, 100), (400, 300, 200, 100)],
     'clf__alpha': [0.0001, 0.001, 0.01, 0.1],
     'clf__max_iter': [3000, 5000, 7000]},  # Neural Network
    {'pca__n_components': [2, 4, 6],  # 主成分数量
     'clf__learning_rate': [0.01, 0.1, 0.5],
     'clf__n_estimators': [50, 100, 200],
     'clf__max_depth': [1, 2, 3],
     'clf__min_child_weight': [1, 3, 5],
     'clf__booster': ['gbtree', 'gblinear']},  # XGBoost
    {'pca__n_components': [2, 4, 6],  # 主成分数量
     'clf__n_estimators': [400, 500, 600],
     'clf__max_depth': [2, 6, 10],
     'clf__min_samples_split': [10, 15, 20],
     'clf__min_samples_leaf': [4, 6, 8],
     'clf__max_features': ['auto', 'sqrt'],
     'clf__bootstrap': [True, False]},  # Random Forest
    {'pca__n_components': [2, 4, 6],  # 主成分数量
     'clf__hidden_layer_sizes': [(50,), (100,), (200,), (100, 50), (200, 100), (300, 200, 100), (400, 300, 200, 100)],
     'clf__alpha': [0.0001, 0.001, 0.01, 0.1],
     'clf__max_iter': [3000, 5000, 7000]}  # Neural Network
]

# 初始化一个空列表，用于保存每个模型的 corr_k
all_corr_k = []

# 对每个模型进行 GridSearchCV
for pipe_name, pipeline in pipelines:
    param_grid = param_grids.pop(0)  # 弹出对应的参数网格
    gs = GridSearchCV(estimator=pipeline,
                      param_grid=param_grid,
                      scoring=['r2', 'neg_mean_absolute_error'],  # use multiple scorers
                      refit='r2',  # use r2 for refitting the best model
                      cv=5,
                      n_jobs=-1)
    gs = gs.fit(X_train, y_train)
    print(f'For {pipe_name}: Best params: {gs.best_params_}, Best score: {gs.best_score_}', "\n")

    # 使用找到的最优模型进行预测，并绘图
    plt.figure(figsize=(60, 40))
    r_score_k = np.zeros((1, 33))
    corr_k = np.zeros((1, 33))
    test_X = []
    for k in range(33):
        test_X = X_test[np.where(well_ind_test == k), :].reshape(-1, 9)
        test_Y = y_test[np.where((well_ind_test == k))]
        test_Depth = DEPTH_test[np.where((well_ind_test == k))]

        train_X = X_test[np.where(well_ind_test != k), :].reshape(-1, 9)
        train_Y = y_test[np.where(well_ind_test != k)]
        train_Depth = DEPTH_test[np.where(well_ind_test != k)]

        gs.best_estimator_.fit(X_train, y_train)
        y_predict = gs.best_estimator_.predict(test_X)
        y_predict[y_predict < 0] = 0

        indicies_0prod = np.where((test_X[:, 6] == 0) | (test_X[:, 7] == 0) | (test_X[:, 8] == 100))
        y_predict[indicies_0prod] = 0

        y_predict_cal = y_predict[test_Y < 2000]  # 只在小值里计算R2
        test_Y_cal = test_Y[test_Y < 2000]
        # 计算R2和皮尔森相关系数
        r_score_k[0, k] = r2_score(test_Y_cal, y_predict_cal)
        corr_k[0, k] = stats.pearsonr(test_Y_cal, y_predict_cal)[0]

        temp_Ymax = np.max([np.max(test_Y), np.max(y_predict)])
        temp_Ymin = np.min([np.min(test_Y), np.min(y_predict)])

        # 把一维数组转为二维数组
        test_Depth = test_Depth.reshape(-1, 1)
        y_predict = y_predict.reshape(-1, 1)
        test_Y = test_Y.reshape(-1, 1)

        # 合并两个矩阵为（9，2）矩阵
        combined = np.concatenate((test_Depth, y_predict, test_Y), axis=1)

        # 使用pandas的DataFrame进行排序
        df = pd.DataFrame(combined, columns=["test_Depth", "y_predict", "test_Y"])
        df = df.sort_values(by="test_Depth")

        # 提取排序后的数据，并转换回numpy数组
        test_Depth_new = df["test_Depth"].values.reshape(-1, 1)
        y_predict_new = df["y_predict"].values.reshape(-1, 1)
        test_Y_new = df["test_Y"].values.reshape(-1, 1)

        tk = plt.subplot(3, 11, k + 1)
        plt.subplots_adjust(hspace=0.25, wspace=0.60)
        plt.plot(test_Y_new, test_Depth_new, c='k', linewidth=4)
        plt.plot(y_predict_new, test_Depth_new, c='r', linewidth=4)
        plt.tick_params(labelsize=30)
        plt.xlabel('real production ($\mathregular{m^3}$/d)',
                   fontdict={'family': 'Times New Roman', 'size': 28, 'weight': 'bold'})
        plt.ylabel('Depth (m)', fontdict={'family': 'Times New Roman', 'size': 28, 'weight': 'bold'})
        plt.yticks(fontproperties='Times New Roman', size=22)
        plt.xticks(fontproperties='Times New Roman', size=22)
        plt.xlim([-5, temp_Ymax + 0.1 * (temp_Ymax - temp_Ymin)])
        plt.ylim([np.min(test_Depth), np.max(test_Depth)])
        plt.text(-0.55 * (temp_Ymax - 0), np.min(test_Depth), '(' + str(k + 1) + ')',
                 fontproperties='Times New Roman', size=28, weight='bold')
        ax = plt.gca()
        ax.invert_yaxis()
        tk.spines['bottom'].set_linewidth(3)
        tk.spines['top'].set_linewidth(3)
        tk.spines['left'].set_linewidth(3)
        tk.spines['right'].set_linewidth(3)

    print(np.mean(r_score_k))
    print(np.mean(corr_k))
    print(np.round(r_score_k, 3))
    print(np.round(corr_k, 3))
    all_corr_k.append(np.squeeze(corr_k))
    plt.savefig(f'D:/LsR/可压性项目产能预测/Fig5_{pipe_name}.svg', dpi=300, bbox_inches="tight")

    # 计算并打印回归指标
    y_pred = gs.best_estimator_.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)  # 计算RMSE
    std_dev = np.std(y_test - y_pred)  # 计算标准差
    std_dev_ref = np.std(y_test)
    std_dev_model = np.std(y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f'MSE: {mse:.2f}')
    print(f'RMSE: {rmse:.2f}')  # 打印RMSE
    print(f'Standard Deviation: {std_dev:.2f}')  # 打印标准差
    print(f'Reference Standard Deviation: {std_dev_ref:.2f}')
    print(f'Model Standard Deviation: {std_dev_model:.2f}')
    print(f'MAE: {mae:.2f}')
    print(f'R^2: {r2:.2f}')

    # 绘制实际值与预测值散点图
    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, y_pred, alpha=0.3)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
    plt.xlabel('Real production(m$^{3}$/d)')
    plt.ylabel('Prediction(m$^{3}$/d)')
    plt.xlim(0, y_test.max())
    plt.ylim(0, y_test.max())
    #plt.title('Actual vs Predicted Values')
    plt.savefig(f'D:/LsR/可压性项目产能预测/Fig6_{pipe_name}.svg', dpi=300, bbox_inches="tight")
    plt.show()

# 将列表转换为一个numpy数组
all_corr_k = np.array(all_corr_k)
# 转置数组，使得每行对应一个well，每列对应一个模型
all_corr_k = all_corr_k.transpose()
# 将数组转换为pandas DataFrame，列名为模型的名称
df_corr_k = pd.DataFrame(all_corr_k, columns=['ps-xgb', 'ps-rf', 'ps-nn','pfs-xgb', 'pfs-rf', 'pfs-nn','pr-xgb', 'pr-rf', 'pr-nn' ])
# 创建热力图，annot参数意味着将在每个格子上显示数值，fmt参数设置了数值的格式
plt.figure(figsize=(8, 12))
heatmap = sns.heatmap(df_corr_k, annot=True, fmt=".3f", cmap="gist_rainbow", cbar=False)
plt.title('Pearson Correlation Coefficients of Different Models on Different Wells')
plt.xlabel('Model')
plt.ylabel('Well index')
# 创建colorbar并放置在底部
cax = plt.gcf().add_axes([0.25, 0.05, 0.55, 0.02])
cbar = plt.gcf().colorbar(heatmap.get_children()[0], cax=cax, orientation='horizontal')
cbar.set_label('Correlation Coefficient')
# 导出为SVG格式
plt.savefig(f'D:/LsR/可压性项目产能预测/Fig7_pcc.svg', dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.savefig(f'D:/LsR/可压性项目产能预测/Fig5_{pipe_name}.svg', dpi=300, bbox_inches="tight")
plt.savefig(f'D:/LsR/可压性项目产能预测/Fig6_{pipe_name}.svg', dpi=300, bbox_inches="tight")
plt.savefig(f'D:/LsR/可压性项目产能预测/Fig7_pcc.svg', dpi=300, bbox_inches="tight")

In [ ]:
df_corr_k = pd.DataFrame(all_corr_k, columns=['ps-xgb', 'ps-rf', 'ps-nn','pfs-xgb', 'pfs-rf', 'pfs-nn','pr-xgb', 'pr-rf', 'pr-nn' ])

plt.figure(figsize=(8, 12))
heatmap = sns.heatmap(df_corr_k, annot=True, fmt=".3f", cmap="RdBu_r", cbar=False)

plt.title('Pearson Correlation Coefficients of Different Models on Different Wells')
plt.xlabel('Model')
plt.ylabel('Well index')                                                                     

# 创建colorbar并放置在底部
cax = plt.gcf().add_axes([0.25, 0.05, 0.55, 0.02])
cbar = plt.gcf().colorbar(heatmap.get_children()[0], cax=cax, orientation='horizontal')
cbar.set_label('Correlation Coefficient')

# 导出为SVG格式
plt.savefig(f'D:/LsR/可压性项目产能预测/Fig7_pcc.svg', dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 将列表转换为一个numpy数组
all_corr_k = np.array(all_corr_k)
# 转置数组，使得每行对应一个well，每列对应一个模型
all_corr_k = all_corr_k.transpose()
# 将数组转换为pandas DataFrame，列名为模型的名称
df_corr_k = pd.DataFrame(all_corr_k, columns=['ps-xgb', 'ps-rf', 'ps-nn','pfs-xgb', 'pfs-rf', 'pfs-nn','pr-xgb', 'pr-rf', 'pr-nn' ])
# 创建热力图，annot参数意味着将在每个格子上显示数值，fmt参数设置了数值的格式
plt.figure(figsize=(8, 12))
heatmap = sns.heatmap(df_corr_k, annot=True, fmt=".3f", cmap="gist_rainbow", cbar=False)
plt.title('Pearson Correlation Coefficients of Different Models on Different Wells')
plt.xlabel('Model')
plt.ylabel('Well index')
# 创建colorbar并放置在底部
cax = plt.gcf().add_axes([0.25, 0.05, 0.55, 0.02])
cbar = plt.gcf().colorbar(heatmap.get_children()[0], cax=cax, orientation='horizontal')
cbar.set_label('Correlation Coefficient')
# 导出为SVG格式
plt.savefig(f'D:/LsR/可压性项目产能预测/Fig7_pcc.svg', dpi=300, bbox_inches="tight")
plt.show()

# 新的一个

In [ ]:
%matplotlib inline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy import stats
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from joblib import dump

In [ ]:
Data_all = pd.read_excel("E:/jupyter lab/可压性项目产能预测-工作站结果/Data_all_forblindtest.xlsx")
feature_name = ['DEPTH','DT','ZDEN','GR','CNCF','M2RX','POR','PERM','SW']
X = Data_all[feature_name].values
y = Data_all['PROD'].values
y[y<0] = 0
X[X[:,6]<0,6] = 0
X[X[:,7]<0,7] = 0
X[X[:,8]>100,8] = 100
well_ind = Data_all['well_ind'].values
DEPTH = Data_all['DEPTH'].values

In [ ]:
# 划分训练集和测试集
X_train, X_test, y_train, y_test, well_ind_train, well_ind_test, DEPTH_train, DEPTH_test = train_test_split(X, y, well_ind, DEPTH, test_size=0.2, random_state=42)

# 定义 XGBoost，随机森林以及神经网络的 pipeline
pipelines = [
    ('ps-xgb', Pipeline([('ss', StandardScaler()),
                         ('pca', PCA()),
                         ('xgb', XGBRegressor(random_state=42))])),
    ('ps-rf', Pipeline([('ss', StandardScaler()),
                        ('pca', PCA()),
                        ('rf', RandomForestRegressor(random_state=42))])),
    ('ps-nn', Pipeline([('ss', StandardScaler()),
                        ('pca', PCA()),
                        ('mlp', MLPRegressor(random_state=42))])),
    ('pfs-xgb', Pipeline([('ss', StandardScaler()),
                          ('poly', PolynomialFeatures()),
                          ('xgb', XGBRegressor(random_state=42))])),
    ('pfs-rf', Pipeline([('ss', StandardScaler()),
                         ('poly', PolynomialFeatures()),
                         ('rf', RandomForestRegressor(random_state=42))])),
    ('pfs-nn', Pipeline([('ss', StandardScaler()),
                         ('poly', PolynomialFeatures()),
                         ('mlp', MLPRegressor(random_state=42))])),
    ('pr-xgb', Pipeline([('rs', RobustScaler()),
                         ('pca', PCA()),
                         ('xgb', XGBRegressor(random_state=42))])),
    ('pr-rf', Pipeline([('rs', RobustScaler()),
                        ('pca', PCA()),
                        ('rf', RandomForestRegressor(random_state=42))])),
    ('pr-nn', Pipeline([('rs', RobustScaler()),
                        ('pca', PCA()),
                        ('mlp', MLPRegressor(random_state=42))])),
    ('pfr-xgb', Pipeline([('rs', RobustScaler()),
                          ('poly', PolynomialFeatures()),
                          ('xgb', XGBRegressor(random_state=42))])),
    ('pfr-rf', Pipeline([('rs', RobustScaler()),
                         ('poly', PolynomialFeatures()),
                         ('rf', RandomForestRegressor(random_state=42))])),
    ('pfr-nn', Pipeline([('rs', RobustScaler()),
                         ('poly', PolynomialFeatures()),
                         ('mlp', MLPRegressor(random_state=42))]))
]

param_grids = [
    # ps-xgb 参数
    {'pca__n_components': [2, 4, 6],
     'xgb__learning_rate': [0.01, 0.1, 0.5],
     'xgb__n_estimators': [50, 100, 200],
     'xgb__max_depth': [1, 2, 3],
     'xgb__min_child_weight': [1, 3, 5],
     'xgb__booster': ['gbtree', 'gblinear']},
    # ps-rf 参数
    {'pca__n_components': [2, 4, 6],
     'rf__n_estimators': [400, 500, 600],
     'rf__max_depth': [2, 6, 10],
     'rf__min_samples_split': [10, 15, 20],
     'rf__min_samples_leaf': [4, 6, 8],
     'rf__max_features': ['auto', 'sqrt'],
     'rf__bootstrap': [True, False]},
    # ps-nn 参数
    {'pca__n_components': [2, 4, 6],
     'mlp__hidden_layer_sizes': [ (100, 50), (200, 100), (300, 200, 100), (400, 300, 200, 100)],
     'mlp__activation': ['tanh', 'relu', 'logistic'],
     'mlp__alpha': [0.0001, 0.001, 0.01, 0.1],
     'mlp__max_iter': [3000, 5000, 7000]},
    # pfs-xgb 参数
    {'poly__degree': [2, 3, 4],
     'xgb__learning_rate': [0.01, 0.1, 0.5],
     'xgb__n_estimators': [50, 100, 200],
     'xgb__max_depth': [1, 2, 3],
     'xgb__min_child_weight': [1, 3, 5],
     'xgb__booster': ['gbtree', 'gblinear']},
    # pfs-rf 参数
    {'poly__degree': [2, 3, 4],
     'rf__n_estimators': [400, 500, 600],
     'rf__max_depth': [2, 6, 10],
     'rf__min_samples_split': [10, 15, 20],
     'rf__min_samples_leaf': [4, 6, 8],
     'rf__max_features': ['auto', 'sqrt'],
     'rf__bootstrap': [True, False]},
    # pfs-nn 参数
    {'poly__degree': [2, 3, 4],
     'mlp__hidden_layer_sizes': [ (100, 50), (200, 100), (300, 200, 100), (400, 300, 200, 100)],
     'mlp__activation': ['tanh', 'relu', 'logistic'],
     'mlp__alpha': [0.0001, 0.001, 0.01, 0.1],
     'mlp__max_iter': [3000, 5000, 7000]},
    # pr-xgb 参数
    {'pca__n_components': [2, 4, 6],
     'xgb__learning_rate': [0.01, 0.1, 0.5],
     'xgb__n_estimators': [50, 100, 200],
     'xgb__max_depth': [1, 2, 3],
     'xgb__min_child_weight': [1, 3, 5],
     'xgb__booster': ['gbtree', 'gblinear']},
    # pr-rf 参数
    {'pca__n_components': [2, 4, 6],
     'rf__n_estimators': [400, 500, 600],
     'rf__max_depth': [2, 6, 10],
     'rf__min_samples_split': [10, 15, 20],
     'rf__min_samples_leaf': [4, 6, 8],
     'rf__max_features': ['auto', 'sqrt'],
     'rf__bootstrap': [True, False]},
    # pr-nn 参数
    {'pca__n_components': [2, 4, 6],
     'mlp__hidden_layer_sizes': [(100, 50), (200, 100), (300, 200, 100), (400, 300, 200, 100)],
     'mlp__activation': ['tanh', 'relu', 'logistic'],
     'mlp__alpha': [0.0001, 0.001, 0.01, 0.1],
     'mlp__max_iter': [3000, 5000, 7000],
     'mlp__early_stopping': [True, False],
     'mlp__validation_fraction': [0.1, 0.2, 0.3]},
    # pfr-xgb 参数
    {'poly__degree': [2, 3, 4],
     'xgb__learning_rate': [0.01, 0.1, 0.5],
     'xgb__n_estimators': [50, 100, 200],
     'xgb__max_depth': [1, 2, 3],
     'xgb__min_child_weight': [1, 3, 5],
     'xgb__booster': ['gbtree', 'gblinear']},
    # pfr-rf 参数
    {'poly__degree': [2, 3, 4],
     'rf__n_estimators': [400, 500, 600],
     'rf__max_depth': [2, 6, 10],
     'rf__min_samples_split': [10, 15, 20],
     'rf__min_samples_leaf': [4, 6, 8],
     'rf__max_features': ['auto', 'sqrt'],
     'rf__bootstrap': [True, False]},
    # pfr-nn 参数
    {'poly__degree': [2, 3, 4],
     'mlp__hidden_layer_sizes': [ (100, 50), (200, 100), (300, 200, 100), (400, 300, 200, 100)],
     'mlp__activation': ['tanh', 'relu', 'logistic'],
     'mlp__alpha': [0.0001, 0.001, 0.01, 0.1],
     'mlp__max_iter': [3000, 5000, 7000]},
]

# 初始化一个空列表，用于保存每个模型的 corr_k
all_corr_k = []

# 对每个模型进行 GridSearchCV
for pipe_name, pipeline in pipelines:
    param_grid = param_grids.pop(0)  # 弹出对应的参数网格
    gs = GridSearchCV(estimator=pipeline,
                      param_grid=param_grid,
                      scoring=['r2', 'neg_mean_absolute_error'],  # use multiple scorers
                      refit='r2',  # use r2 for refitting the best model
                      cv=5,
                      n_jobs=-1)
    gs = gs.fit(X_train, y_train)
    print(f'For {pipe_name}: Best params: {gs.best_params_}, Best score: {gs.best_score_}', "\n")
    
    # 保存训练好的最优模型
    model_filename = f'E:/jupyter lab/可压性项目产能预测/models/best_model_{pipe_name}.joblib'
    dump(gs.best_estimator_, model_filename)
    
    # 使用找到的最优模型进行预测，并绘图
    plt.figure(figsize=(60, 40))
    r_score_k = np.zeros((1, 33))
    corr_k = np.zeros((1, 33))
    test_X = []
    for k in range(33):
        test_X = X_test[np.where(well_ind_test == k), :].reshape(-1, 9)
        test_Y = y_test[np.where((well_ind_test == k))]
        test_Depth = DEPTH_test[np.where((well_ind_test == k))]

        train_X = X_test[np.where(well_ind_test != k), :].reshape(-1, 9)
        train_Y = y_test[np.where(well_ind_test != k)]
        train_Depth = DEPTH_test[np.where(well_ind_test != k)]

        y_predict = gs.best_estimator_.predict(test_X)
        y_predict[y_predict < 0] = 0

        indicies_0prod = np.where((test_X[:, 6] == 0) | (test_X[:, 7] == 0) | (test_X[:, 8] == 100))
        y_predict[indicies_0prod] = 0

        y_predict_cal = y_predict[test_Y < 2000]  # 只在小值里计算R2
        test_Y_cal = test_Y[test_Y < 2000]
        # 计算R2和皮尔森相关系数
        r_score_k[0, k] = r2_score(test_Y_cal, y_predict_cal)
        corr_k[0, k] = stats.pearsonr(test_Y_cal, y_predict_cal)[0]

        temp_Ymax = np.max([np.max(test_Y), np.max(y_predict)])
        temp_Ymin = np.min([np.min(test_Y), np.min(y_predict)])

        # 把一维数组转为二维数组
        test_Depth = test_Depth.reshape(-1, 1)
        y_predict = y_predict.reshape(-1, 1)
        test_Y = test_Y.reshape(-1, 1)

        # 合并两个矩阵为（9，2）矩阵
        combined = np.concatenate((test_Depth, y_predict, test_Y), axis=1)

        # 使用pandas的DataFrame进行排序
        df = pd.DataFrame(combined, columns=["test_Depth", "y_predict", "test_Y"])
        df = df.sort_values(by="test_Depth")

        # 提取排序后的数据，并转换回numpy数组
        test_Depth_new = df["test_Depth"].values.reshape(-1, 1)
        y_predict_new = df["y_predict"].values.reshape(-1, 1)
        test_Y_new = df["test_Y"].values.reshape(-1, 1)

        tk = plt.subplot(3, 11, k + 1)
        plt.subplots_adjust(hspace=0.25, wspace=0.60)
        plt.plot(test_Y_new, test_Depth_new, c='k', linewidth=4)
        plt.plot(y_predict_new, test_Depth_new, c='r', linewidth=4)
        plt.tick_params(labelsize=30)
        plt.xlabel('real production ($\mathregular{m^3}$/d)',
                   fontdict={'family': 'Times New Roman', 'size': 28, 'weight': 'bold'})
        plt.ylabel('Depth (m)', fontdict={'family': 'Times New Roman', 'size': 28, 'weight': 'bold'})
        plt.yticks(fontproperties='Times New Roman', size=22)
        plt.xticks(fontproperties='Times New Roman', size=22)
        plt.xlim([-5, temp_Ymax + 0.1 * (temp_Ymax - temp_Ymin)])
        plt.ylim([np.min(test_Depth), np.max(test_Depth)])
        plt.text(-0.55 * (temp_Ymax - 0), np.min(test_Depth), '(' + str(k + 1) + ')',
                 fontproperties='Times New Roman', size=28, weight='bold')
        ax = plt.gca()
        ax.invert_yaxis()
        tk.spines['bottom'].set_linewidth(3)
        tk.spines['top'].set_linewidth(3)
        tk.spines['left'].set_linewidth(3)
        tk.spines['right'].set_linewidth(3)

    print(f'Average R² Score: {np.mean(r_score_k)}')
    print(f'Average Pearson Correlation Coefficient: {np.mean(corr_k)}')
    print(f'R² Scores per Test Sample : {np.round(r_score_k, 3)}')
    print(f'Pearson Correlation Coefficients per Test Sample : {np.round(corr_k, 3)}')
    all_corr_k.append(np.squeeze(corr_k))
    plt.savefig(f'E:/jupyter lab/可压性项目产能预测/photo/Fig5_{pipe_name}.svg', dpi=300, bbox_inches="tight")

    # 计算并打印回归指标
    y_pred = gs.best_estimator_.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)  # 计算RMSE
    std_dev = np.std(y_test - y_pred)  # 计算标准差
    std_dev_ref = np.std(y_test)
    std_dev_model = np.std(y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f'MSE: {mse:.2f}')
    print(f'RMSE: {rmse:.2f}')  # 打印RMSE
    print(f'Standard Deviation: {std_dev:.2f}')  # 打印标准差
    print(f'Reference Standard Deviation: {std_dev_ref:.2f}')
    print(f'Model Standard Deviation: {std_dev_model:.2f}')
    print(f'MAE: {mae:.2f}')
    print(f'R^2: {r2:.2f}')

    # 绘制实际值与预测值散点图
    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, y_pred, alpha=0.3)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
    plt.xlabel('Real production(m$^{3}$/d)')
    plt.ylabel('Prediction(m$^{3}$/d)')
    plt.xlim(0, y_test.max())
    plt.ylim(0, y_test.max())
    #plt.title('Actual vs Predicted Values')
    plt.savefig(f'E:/jupyter lab/可压性项目产能预测/photo/Fig6_{pipe_name}.svg', dpi=300, bbox_inches="tight")
    plt.show()

# 将列表转换为一个numpy数组
all_corr_k = np.array(all_corr_k)
# 转置数组，使得每行对应一个well，每列对应一个模型
all_corr_k = all_corr_k.transpose()
# 创建一个从 1 到 33 的行索引列表
well_indices = list(range(1, 34))
# 将数组转换为 pandas DataFrame，列名为模型的名称，行索引为 well_indices
df_corr_k = pd.DataFrame(all_corr_k, index=well_indices, columns=['ps-xgb', 'ps-rf', 'ps-nn', 'pfs-xgb', 'pfs-rf', 'pfs-nn', 'pr-xgb', 'pr-rf', 'pr-nn', 'pfr-xgb', 'pfr-rf', 'pfr-nn'])

# 绘制热图
plt.figure(figsize=(10, 12))
heatmap = sns.heatmap(df_corr_k, annot=True, fmt=".3f", cmap="RdBu_r", cbar=False)
plt.title('Pearson Correlation Coefficients of Different Models on Different Wells')
plt.xlabel('Model')
plt.ylabel('Well index')

# 创建colorbar并放置在底部
cax = plt.gcf().add_axes([0.25, 0.05, 0.55, 0.02])
cbar = plt.gcf().colorbar(heatmap.get_children()[0], cax=cax, orientation='horizontal')
cbar.set_label('Correlation Coefficient')

# 导出为SVG格式
plt.savefig(f'E:/jupyter lab/可压性项目产能预测/photo/Fig7_pcc.svg', dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 创建一个从 1 到 33 的行索引列表
well_indices = list(range(1, 34))

# 将数组转换为 pandas DataFrame，列名为模型的名称，行索引为 well_indices
df_corr_k = pd.DataFrame(all_corr_k, index=well_indices, columns=['ps-xgb', 'ps-rf', 'ps-nn', 'pfs-xgb', 'pfs-rf', 'pfs-nn', 'pr-xgb', 'pr-rf', 'pr-nn', 'pfr-xgb', 'pfr-rf', 'pfr-nn'])

# 绘制热图
plt.figure(figsize=(10, 12))
heatmap = sns.heatmap(df_corr_k, annot=True, fmt=".3f", cmap="RdBu_r", cbar=False)

plt.title('Pearson Correlation Coefficients of Different Models on Different Wells')
plt.xlabel('Model')
plt.ylabel('Well Index')

# 创建 colorbar 并放置在底部
cax = plt.gcf().add_axes([0.25, 0.05, 0.55, 0.02])
cbar = plt.gcf().colorbar(heatmap.get_children()[0], cax=cax, orientation='horizontal')
cbar.set_label('Pearson Correlation Coefficient')

# 导出为 SVG 格式
plt.savefig(f'E:/jupyter lab/可压性项目产能预测/photo/Fig7_pcc.svg', dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 划分训练集和测试集
X_train, X_test, y_train, y_test, well_ind_train, well_ind_test, DEPTH_train, DEPTH_test = train_test_split(X, y, well_ind, DEPTH, test_size=0.2, random_state=42)

# 定义 XGBoost，随机森林以及神经网络的 pipeline
pipelines = [
    ('ps-xgb', Pipeline([('ss', StandardScaler()),
                         ('pca', PCA()),
                         ('xgb', XGBRegressor(random_state=42))])),
    ('ps-rf', Pipeline([('ss', StandardScaler()),
                        ('pca', PCA()),
                        ('rf', RandomForestRegressor(random_state=42))])),
    ('ps-nn', Pipeline([('ss', StandardScaler()),
                        ('pca', PCA()),
                        ('mlp', MLPRegressor(random_state=42))])),
    ('pfs-xgb', Pipeline([('ss', StandardScaler()),
                          ('poly', PolynomialFeatures()),
                          ('xgb', XGBRegressor(random_state=42))])),
    ('pfs-rf', Pipeline([('ss', StandardScaler()),
                         ('poly', PolynomialFeatures()),
                         ('rf', RandomForestRegressor(random_state=42))])),
    ('pfs-nn', Pipeline([('ss', StandardScaler()),
                         ('poly', PolynomialFeatures()),
                         ('mlp', MLPRegressor(random_state=42))])),
    ('pr-xgb', Pipeline([('rs', RobustScaler()),
                         ('pca', PCA()),
                         ('xgb', XGBRegressor(random_state=42))])),
    ('pr-rf', Pipeline([('rs', RobustScaler()),
                        ('pca', PCA()),
                        ('rf', RandomForestRegressor(random_state=42))])),
    ('pr-nn', Pipeline([('rs', RobustScaler()),
                        ('pca', PCA()),
                        ('mlp', MLPRegressor(random_state=42))])),
    ('pfr-xgb', Pipeline([('rs', RobustScaler()),
                          ('poly', PolynomialFeatures()),
                          ('xgb', XGBRegressor(random_state=42))])),
    ('pfr-rf', Pipeline([('rs', RobustScaler()),
                         ('poly', PolynomialFeatures()),
                         ('rf', RandomForestRegressor(random_state=42))])),
    ('pfr-nn', Pipeline([('rs', RobustScaler()),
                         ('poly', PolynomialFeatures()),
                         ('mlp', MLPRegressor(random_state=42))]))
]

param_grids = [
    # ps-xgb 参数
    {'pca__n_components': [2, 4, 6],
     'xgb__learning_rate': [0.01, 0.1, 0.5],
     'xgb__n_estimators': [50, 100, 200],
     'xgb__max_depth': [1, 2, 3],
     'xgb__min_child_weight': [1, 3, 5],
     'xgb__booster': ['gbtree', 'gblinear']},
    # ps-rf 参数
    {'pca__n_components': [2, 4, 6],
     'rf__n_estimators': [400, 500, 600],
     'rf__max_depth': [2, 6, 10],
     'rf__min_samples_split': [10, 15, 20],
     'rf__min_samples_leaf': [4, 6, 8],
     'rf__max_features': ['auto', 'sqrt'],
     'rf__bootstrap': [True, False]},
    # ps-nn 参数
    {'pca__n_components': [2, 4, 6],
     'mlp__hidden_layer_sizes': [ (100, 50), (200, 100), (300, 200, 100), (400, 300, 200, 100)],
     'mlp__activation': ['tanh', 'relu', 'logistic'],
     'mlp__alpha': [0.0001, 0.001, 0.01, 0.1],
     'mlp__max_iter': [3000, 5000, 7000]},
    # pfs-xgb 参数
    {'poly__degree': [2, 3, 4],
     'xgb__learning_rate': [0.01, 0.1, 0.5],
     'xgb__n_estimators': [50, 100, 200],
     'xgb__max_depth': [1, 2, 3],
     'xgb__min_child_weight': [1, 3, 5],
     'xgb__booster': ['gbtree', 'gblinear']},
    # pfs-rf 参数
    {'poly__degree': [2, 3, 4],
     'rf__n_estimators': [400, 500, 600],
     'rf__max_depth': [2, 6, 10],
     'rf__min_samples_split': [10, 15, 20],
     'rf__min_samples_leaf': [4, 6, 8],
     'rf__max_features': ['auto', 'sqrt'],
     'rf__bootstrap': [True, False]},
    # pfs-nn 参数
    {'poly__degree': [2, 3, 4],
     'mlp__hidden_layer_sizes': [ (100, 50), (200, 100), (300, 200, 100), (400, 300, 200, 100)],
     'mlp__activation': ['tanh', 'relu', 'logistic'],
     'mlp__alpha': [0.0001, 0.001, 0.01, 0.1],
     'mlp__max_iter': [3000, 5000, 7000]},
    # pr-xgb 参数
    {'pca__n_components': [2, 4, 6],
     'xgb__learning_rate': [0.01, 0.1, 0.5],
     'xgb__n_estimators': [50, 100, 200],
     'xgb__max_depth': [1, 2, 3],
     'xgb__min_child_weight': [1, 3, 5],
     'xgb__booster': ['gbtree', 'gblinear']},
    # pr-rf 参数
    {'pca__n_components': [2, 4, 6],
     'rf__n_estimators': [400, 500, 600],
     'rf__max_depth': [2, 6, 10],
     'rf__min_samples_split': [10, 15, 20],
     'rf__min_samples_leaf': [4, 6, 8],
     'rf__max_features': ['auto', 'sqrt'],
     'rf__bootstrap': [True, False]},
    # pr-nn 参数
    {'pca__n_components': [2, 4, 6],
     'mlp__hidden_layer_sizes': [(100, 50), (200, 100), (300, 200, 100), (400, 300, 200, 100)],
     'mlp__activation': ['tanh', 'relu', 'logistic'],
     'mlp__alpha': [0.0001, 0.001, 0.01, 0.1],
     'mlp__max_iter': [3000, 5000, 7000],
     'mlp__early_stopping': [True, False],
     'mlp__validation_fraction': [0.1, 0.2, 0.3]},
    # pfr-xgb 参数
    {'poly__degree': [2, 3, 4],
     'xgb__learning_rate': [0.01, 0.1, 0.5],
     'xgb__n_estimators': [50, 100, 200],
     'xgb__max_depth': [1, 2, 3],
     'xgb__min_child_weight': [1, 3, 5],
     'xgb__booster': ['gbtree', 'gblinear']},
    # pfr-rf 参数
    {'poly__degree': [2, 3, 4],
     'rf__n_estimators': [400, 500, 600],
     'rf__max_depth': [2, 6, 10],
     'rf__min_samples_split': [10, 15, 20],
     'rf__min_samples_leaf': [4, 6, 8],
     'rf__max_features': ['auto', 'sqrt'],
     'rf__bootstrap': [True, False]},
    # pfr-nn 参数
    {'poly__degree': [2, 3, 4],
     'mlp__hidden_layer_sizes': [ (100, 50), (200, 100), (300, 200, 100), (400, 300, 200, 100)],
     'mlp__activation': ['tanh', 'relu', 'logistic'],
     'mlp__alpha': [0.0001, 0.001, 0.01, 0.1],
     'mlp__max_iter': [3000, 5000, 7000]},
]

# 初始化一个空列表，用于保存每个模型的 corr_k
all_corr_k = []

# 对每个模型进行 GridSearchCV
for pipe_name, pipeline in pipelines:
    param_grid = param_grids.pop(0)  # 弹出对应的参数网格
    gs = GridSearchCV(estimator=pipeline,
                      param_grid=param_grid,
                      scoring=['r2', 'neg_mean_absolute_error'],  # use multiple scorers
                      refit='r2',  # use r2 for refitting the best model
                      cv=5,
                      n_jobs=-1)
    gs = gs.fit(X_train, y_train)
    print(f'For {pipe_name}: Best params: {gs.best_params_}, Best score: {gs.best_score_}', "\n")

    # 保存训练好的最优模型
    model_filename = f'E:/jupyter lab/可压性项目产能预测/models/best_model_{pipe_name}.joblib'
    dump(gs.best_estimator_, model_filename)

    # 使用找到的最优模型进行预测，并绘图
    plt.figure(figsize=(60, 40))
    r_score_k = np.zeros((1, 33))
    corr_k = np.zeros((1, 33))
    test_X = []
    for k in range(33):
        test_X = X_test[np.where(well_ind_test == k), :].reshape(-1, 9)
        test_Y = y_test[np.where((well_ind_test == k))]
        test_Depth = DEPTH_test[np.where((well_ind_test == k))]

        train_X = X_test[np.where(well_ind_test != k), :].reshape(-1, 9)
        train_Y = y_test[np.where(well_ind_test != k)]
        train_Depth = DEPTH_test[np.where(well_ind_test != k)]

        y_predict = gs.best_estimator_.predict(test_X)
        y_predict[y_predict < 0] = 0

        indicies_0prod = np.where((test_X[:, 6] == 0) | (test_X[:, 7] == 0) | (test_X[:, 8] == 100))
        y_predict[indicies_0prod] = 0

        y_predict_cal = y_predict[test_Y < 2000]  # 只在小值里计算R2
        test_Y_cal = test_Y[test_Y < 2000]
        # 计算R2和皮尔森相关系数
        r_score_k[0, k] = r2_score(test_Y_cal, y_predict_cal)
        corr_k[0, k] = stats.pearsonr(test_Y_cal, y_predict_cal)[0]

        temp_Ymax = np.max([np.max(test_Y), np.max(y_predict)])
        temp_Ymin = np.min([np.min(test_Y), np.min(y_predict)])

        # 把一维数组转为二维数组
        test_Depth = test_Depth.reshape(-1, 1)
        y_predict = y_predict.reshape(-1, 1)
        test_Y = test_Y.reshape(-1, 1)

        # 合并两个矩阵为（9，2）矩阵
        combined = np.concatenate((test_Depth, y_predict, test_Y), axis=1)

        # 使用pandas的DataFrame进行排序
        df = pd.DataFrame(combined, columns=["test_Depth", "y_predict", "test_Y"])
        df = df.sort_values(by="test_Depth")

        # 提取排序后的数据，并转换回numpy数组
        test_Depth_new = df["test_Depth"].values.reshape(-1, 1)
        y_predict_new = df["y_predict"].values.reshape(-1, 1)
        test_Y_new = df["test_Y"].values.reshape(-1, 1)

        tk = plt.subplot(3, 11, k + 1)
        plt.subplots_adjust(hspace=0.25, wspace=0.60)
        plt.scatter(test_Y_new, test_Depth_new, c='blue', marker='o', s=250)  # 绘制实际数据点
        plt.scatter(y_predict_new, test_Depth_new, c='green', marker='s', s=250)        # 绘制预测数据点
        plt.plot(test_Y_new, test_Depth_new, c='k', linewidth=4)
        plt.plot(y_predict_new, test_Depth_new, c='r', linewidth=4)
        plt.tick_params(labelsize=30)
        plt.xlabel('real production ($\mathregular{m^3}$/d)',
                   fontdict={'family': 'Times New Roman', 'size': 28, 'weight': 'bold'})
        plt.ylabel('Depth (m)', fontdict={'family': 'Times New Roman', 'size': 28, 'weight': 'bold'})
        plt.yticks(fontproperties='Times New Roman', size=22)
        plt.xticks(fontproperties='Times New Roman', size=22)
        plt.xlim([-5, temp_Ymax + 0.1 * (temp_Ymax - temp_Ymin)])
        plt.ylim([np.min(test_Depth), np.max(test_Depth)])
        plt.text(-0.55 * (temp_Ymax - 0), np.min(test_Depth), '(' + str(k + 1) + ')',
                 fontproperties='Times New Roman', size=28, weight='bold')
        ax = plt.gca()
        ax.invert_yaxis()
        tk.spines['bottom'].set_linewidth(3)
        tk.spines['top'].set_linewidth(3)
        tk.spines['left'].set_linewidth(3)
        tk.spines['right'].set_linewidth(3)

    print(f'Average R² Score: {np.mean(r_score_k)}')
    print(f'Average Pearson Correlation Coefficient: {np.mean(corr_k)}')
    print(f'R² Scores per Test Sample : {np.round(r_score_k, 3)}')
    print(f'Pearson Correlation Coefficients per Test Sample : {np.round(corr_k, 3)}')
    all_corr_k.append(np.squeeze(corr_k))
    plt.savefig(f'E:/jupyter lab/可压性项目产能预测/photo/Fig5_{pipe_name}.svg', dpi=300, bbox_inches="tight")

    # 计算并打印回归指标
    y_pred = gs.best_estimator_.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)  # 计算RMSE
    std_dev = np.std(y_test - y_pred)  # 计算标准差
    std_dev_ref = np.std(y_test)
    std_dev_model = np.std(y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f'MSE: {mse:.2f}')
    print(f'RMSE: {rmse:.2f}')  # 打印RMSE
    print(f'Standard Deviation: {std_dev:.2f}')  # 打印标准差
    print(f'Reference Standard Deviation: {std_dev_ref:.2f}')
    print(f'Model Standard Deviation: {std_dev_model:.2f}')
    print(f'MAE: {mae:.2f}')
    print(f'R^2: {r2:.2f}')

    # 绘制实际值与预测值散点图
    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, y_pred, alpha=0.3)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
    plt.xlabel('Real production(m$^{3}$/d)')
    plt.ylabel('Prediction(m$^{3}$/d)')
    plt.xlim(0, y_test.max())
    plt.ylim(0, y_test.max())
    #plt.title('Actual vs Predicted Values')
    plt.savefig(f'E:/jupyter lab/可压性项目产能预测/photo/Fig6_{pipe_name}.svg', dpi=300, bbox_inches="tight")
    plt.show()

# 将列表转换为一个numpy数组
all_corr_k = np.array(all_corr_k)
# 转置数组，使得每行对应一个well，每列对应一个模型
all_corr_k = all_corr_k.transpose()
# 创建一个从 1 到 33 的行索引列表
well_indices = list(range(1, 34))

# 将数组转换为 pandas DataFrame，列名为模型的名称，行索引为 well_indices
df_corr_k = pd.DataFrame(all_corr_k, index=well_indices, columns=['ps-xgb', 'ps-rf', 'ps-nn', 'pfs-xgb', 'pfs-rf', 'pfs-nn', 'pr-xgb', 'pr-rf', 'pr-nn', 'pfr-xgb', 'pfr-rf', 'pfr-nn'])

# 绘制热图
plt.figure(figsize=(10, 12))
heatmap = sns.heatmap(df_corr_k, annot=True, fmt=".3f", cmap="RdBu_r", cbar=False)

plt.title('Pearson Correlation Coefficients of Different Models on Different Wells')
plt.xlabel('Model')
plt.ylabel('Well Index')

# 创建 colorbar 并放置在底部
cax = plt.gcf().add_axes([0.25, 0.05, 0.55, 0.02])
cbar = plt.gcf().colorbar(heatmap.get_children()[0], cax=cax, orientation='horizontal')
cbar.set_label('Pearson Correlation Coefficient')

# 导出为 SVG 格式
plt.savefig(f'E:/jupyter lab/可压性项目产能预测/photo/Fig7_pcc.svg', dpi=300, bbox_inches="tight")
plt.show()

# shap训练

In [ ]:
Data_all = pd.read_excel("E:/jupyter lab/可压性项目产能预测-工作站结果/Data_all_forblindtest.xlsx")
feature_name = ['DEPTH','DT','ZDEN','GR','CNCF','M2RX','POR','PERM','SW']
X = Data_all[feature_name].values
y = Data_all['PROD'].values
y[y<0] = 0
X[X[:,6]<0,6] = 0
X[X[:,7]<0,7] = 0
X[X[:,8]>100,8] = 100
well_ind = Data_all['well_ind'].values
DEPTH = Data_all['DEPTH'].values

In [ ]:
X_train, X_test, y_train, y_test, well_ind_train, well_ind_test, DEPTH_train, DEPTH_test = train_test_split(X, y, well_ind, DEPTH, test_size=0.2, random_state=42)

In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt
from joblib import load
import os

# 设置全局字体大小
plt.rcParams.update({
    'font.size': 16,  # 基础字体大小
    'axes.titlesize': 18,  # 标题字体大小
    'axes.labelsize': 16,  # 轴标签字体大小
    'xtick.labelsize': 14,  # x轴刻度标签大小
    'ytick.labelsize': 14,  # y轴刻度标签大小
    'legend.fontsize': 14,  # 图例字体大小
})

def sanitize_filename(name):
    """
    将特征名称转换为合法的文件名
    """
    replacements = {
        '*': 'times',
        '/': 'div',
        ':': '_',
        '?': '_',
        '"': '_',
        '<': 'lt',
        '>': 'gt',
        '|': '_',
        '\\': '_',
        '^': 'pow'
    }

    for old, new in replacements.items():
        name = name.replace(old, new)
    return name

def explain_models_with_shap(model_names, X_train, X_test, feature_names):
    """
    对指定的模型进行SHAP解释
    """
    # 创建保存图片的目录
    save_dir = "E:/jupyter lab/可压性项目产能预测-工作站结果/shap_plots"
    os.makedirs(save_dir, exist_ok=True)

    for model_name in model_names:
        print(f"\n正在为 {model_name} 生成SHAP解释...")

        # 加载保存的模型
        model_path = f'E:/jupyter lab/可压性项目产能预测/models/best_model_{model_name}.joblib'
        pipeline = load(model_path)

        # 获取数据预处理后的特征
        if model_name.startswith(('pfs-', 'pfr-')):
            X_scaled = pipeline.named_steps['ss' if model_name.startswith('pfs-') else 'rs'].transform(X_test)
            X_transformed = pipeline.named_steps['poly'].transform(X_scaled)
            degree = pipeline.named_steps['poly'].degree
            transformed_feature_names = []
            for d in range(1, degree + 1):
                for i in range(len(feature_names)):
                    for j in range(i + 1):
                        if i == j:
                            transformed_feature_names.append(f"{feature_names[i]}^{d}")
                        else:
                            transformed_feature_names.append(f"{feature_names[i]}*{feature_names[j]}")
        else:
            X_scaled = pipeline.named_steps['ss' if model_name.startswith('ps-') else 'rs'].transform(X_test)
            X_transformed = pipeline.named_steps['pca'].transform(X_scaled)
            n_components = pipeline.named_steps['pca'].n_components_
            transformed_feature_names = [f"PC{i+1}" for i in range(n_components)]

        if 'mlp' in pipeline.named_steps:
            # 神经网络模型使用KernelExplainer
            background = shap.kmeans(X_transformed, 50)
            mlp_model = pipeline.named_steps['mlp']
            explainer = shap.KernelExplainer(mlp_model.predict, background)

            sample_size = min(100, len(X_test))
            shap_values = explainer.shap_values(X_transformed[:sample_size])

            # 生成summary plot
            plt.figure(figsize=(15, 10))  # 增大图片尺寸
            shap.summary_plot(shap_values, X_transformed[:sample_size],
                              feature_names=transformed_feature_names,
                              show=False)
            plt.title(f'SHAP Summary Plot - {model_name}', pad=20, fontsize=18)
            plt.tight_layout()
            plt.savefig(os.path.join(save_dir, f'shap_summary_{model_name}.png'), dpi=300, bbox_inches='tight')
            plt.close()

            # 生成feature importance plot
            plt.figure(figsize=(15, 10))
            shap.summary_plot(shap_values, X_transformed[:sample_size],
                              feature_names=transformed_feature_names,
                              plot_type="bar",
                              show=False)
            plt.title(f'SHAP Feature Importance - {model_name}', pad=20, fontsize=18)
            plt.tight_layout()
            plt.savefig(os.path.join(save_dir, f'shap_importance_{model_name}.png'), dpi=300, bbox_inches='tight')
            plt.close()

            # 生成top 4特征的依赖图
            importance = np.abs(shap_values).mean(0)
            top_indices = importance.argsort()[-4:][::-1]  # 修改为top 4

            for idx in top_indices:
                plt.figure(figsize=(12, 8))
                shap.dependence_plot(idx, shap_values, X_transformed[:sample_size],
                                     feature_names=transformed_feature_names,
                                     show=False)
                plt.title(f'SHAP Dependence Plot - {transformed_feature_names[idx]} ({model_name})',
                          pad=20, fontsize=18)
                plt.tight_layout()
                safe_filename = sanitize_filename(transformed_feature_names[idx])
                plt.savefig(os.path.join(save_dir, f'shap_dependence_{model_name}_{safe_filename}.png'),
                            dpi=300, bbox_inches='tight')
                plt.close()

        elif 'xgb' in pipeline.named_steps:
            # XGBoost模型使用TreeExplainer
            xgb_model = pipeline.named_steps['xgb']
            explainer = shap.TreeExplainer(xgb_model)
            shap_values = explainer.shap_values(X_transformed)

            # 生成summary plot
            plt.figure(figsize=(15, 10))
            shap.summary_plot(shap_values, X_transformed,
                              feature_names=transformed_feature_names,
                              show=False)
            plt.title(f'SHAP Summary Plot - {model_name}', pad=20, fontsize=18)
            plt.tight_layout()
            plt.savefig(os.path.join(save_dir, f'shap_summary_{model_name}.png'), dpi=300, bbox_inches='tight')
            plt.close()

            # 生成feature importance plot
            plt.figure(figsize=(15, 10))
            shap.summary_plot(shap_values, X_transformed,
                              feature_names=transformed_feature_names,
                              plot_type="bar",
                              show=False)
            plt.title(f'SHAP Feature Importance - {model_name}', pad=20, fontsize=18)
            plt.tight_layout()
            plt.savefig(os.path.join(save_dir, f'shap_importance_{model_name}.png'), dpi=300, bbox_inches='tight')
            plt.close()

            # 生成top 4特征的依赖图
            importance = np.abs(shap_values).mean(0)
            top_indices = importance.argsort()[-4:][::-1]  # 修改为top 4

            for idx in top_indices:
                plt.figure(figsize=(12, 8))
                shap.dependence_plot(idx, shap_values, X_transformed,
                                     feature_names=transformed_feature_names,
                                     show=False)
                plt.title(f'SHAP Dependence Plot - {transformed_feature_names[idx]} ({model_name})',
                          pad=20, fontsize=18)
                plt.tight_layout()
                safe_filename = sanitize_filename(transformed_feature_names[idx])
                plt.savefig(os.path.join(save_dir, f'shap_dependence_{model_name}_{safe_filename}.png'),
                            dpi=300, bbox_inches='tight')
                plt.close()

# 指定要分析的模型
models_to_explain = ['pfs-nn', 'pfr-nn', 'ps-nn', 'pfs-xgb']

# 运行SHAP解释
explain_models_with_shap(
    models_to_explain,
    X_train,
    X_test,
    feature_names=['DEPTH', 'DT', 'ZDEN', 'GR', 'CNCF', 'M2RX', 'POR', 'PERM', 'SW']
)

In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import os
from joblib import load

# 设置全局字体大小
plt.rcParams.update({
    'font.size': 16,            # 基础字体大小
    'axes.titlesize': 18,       # 标题字体大小
    'axes.labelsize': 16,       # 轴标签字体大小
    'xtick.labelsize': 14,      # x轴刻度标签大小
    'ytick.labelsize': 14,      # y轴刻度标签大小
    'legend.fontsize': 14,      # 图例字体大小
})

def sanitize_filename(name):
    """
    将特征名称转换为合法的文件名
    """
    replacements = {
        '*': 'times',
        '/': 'div',
        ':': '_',
        '?': '_',
        '"': '_',
        '<': 'lt',
        '>': 'gt',
        '|': '_',
        '\\': '_',
        '^': 'pow'
    }
    for old, new in replacements.items():
        name = name.replace(old, new)
    return name

def combined_summary_importance_plot(shap_values, features, feature_names, save_path=None, max_display=20):
    """
    在同一个图层里叠加：
    1. 从左至右显示的条形图 (Feature Importance)，x 轴范围 [0, importance_max * 1.05].
    2. SHAP 蜜蜂群图 (SHAP Value)，x 轴范围 [-shap_max, shap_max]，0 在中间。
    二者共享同一套 y 轴(特征)，并保持顺序一致。
    """
    # -----------------------
    # 1) 排序和基本数据计算
    # -----------------------
    # 计算特征重要性（SHAP值绝对值的均值）
    feature_importance = np.abs(shap_values).mean(axis=0)

    # 限制显示的最大特征数量
    max_display = min(max_display, len(feature_names))
    # 按重要性降序排序
    sorted_indices = np.argsort(feature_importance)[::-1][:max_display]
    sorted_feature_names = np.array(feature_names)[sorted_indices]
    sorted_shap_values = shap_values[:, sorted_indices]
    sorted_features = features[:, sorted_indices]
    sorted_importance = feature_importance[sorted_indices]

    # 最大值
    shap_max = np.max(np.abs(sorted_shap_values))    # SHAP 值最大绝对值
    importance_max = np.max(sorted_importance)       # 条形图最大值

    # -----------------------
    # 2) 创建画布和轴
    # -----------------------
    fig, ax_imp = plt.subplots(figsize=(8, 12))
    # 创建一个共享 y 轴的副轴用于 SHAP Summary Plot
    ax_shap = ax_imp.twiny()

    # -----------------------
    # 3) 绘制 Feature Importance 条形图 (ax_imp)
    # -----------------------
    y_positions = np.arange(len(sorted_feature_names))
    ax_imp.barh(
        y_positions,
        sorted_importance,
        color='lightblue',
        alpha=0.8,
        label='Feature Importance',
        zorder=1  # 确保条形图在底层
    )

    # -----------------------
    # 5) 添加 Mean |SHAP Value| 标签
    # -----------------------
    # 将标签放在 SHAP Summary Plot 的 0 位置附近
    label_offset = importance_max * 0.02  # 调整此值以控制标签的水平偏移
    for i, v in enumerate(sorted_importance):
        ax_imp.text(
            0 + label_offset,  # 在0右侧偏移
            i,
            f"{v:.2f}",
            va='center',
            fontsize=14,
            color='blue',
            ha='left'  # 左对齐
        )

    # 设置条形图 X 轴范围：从 0 到 importance_max * 1.05
    ax_imp.set_xlim(0, importance_max * 1.05)  # 留一点空白，让图更美观
    ax_imp.set_xlabel("Mean |SHAP Value| (Feature Importance)", fontsize=14, color="blue")
    ax_imp.tick_params(axis='x', labelsize=14, colors="blue")

    # 设置 Y 轴：特征名从上到下
    ax_imp.set_yticks(y_positions)
    ax_imp.set_yticklabels(sorted_feature_names, fontsize=14)
    ax_imp.invert_yaxis()  # 最重要特征在最上方

    # -----------------------
    # 4) 绘制 SHAP 蜜蜂群图 (ax_shap)
    # -----------------------
    # 设置 SHAP Summary Plot 的 X 轴范围
    ax_shap.set_xlim(-shap_max, shap_max)
    ax_shap.set_xlabel("SHAP Value (Impact on Model Output)", fontsize=14, color="green")
    ax_shap.tick_params(axis='x', labelsize=14, colors="green")

    # 绘制每个特征的 SHAP 值密度图
    for i in range(len(sorted_feature_names)):
        shap_vals = sorted_shap_values[:, i]
        feature_vals = sorted_features[:, i]
        # 计算密度
        if len(shap_vals) > 1:
            kde = gaussian_kde(shap_vals)
            x_d = np.linspace(-shap_max, shap_max, 200)
            y_d = kde(x_d)
            # 归一化 y_d 并调整宽度
            y_d_normalized = y_d / y_d.max() * 0.4  # 调整此值以控制密度宽度

            # 绘制密度图
            ax_shap.fill_betweenx(
                y=np.full_like(x_d, i),
                x1=x_d - y_d_normalized,
                x2=x_d + y_d_normalized,
                color='grey',
                alpha=0.5,
                zorder=1  # 确保密度图在条形图下方
            )
        else:
            # 如果 SHAP 值数量不足，直接绘制点
            ax_shap.scatter(shap_vals, np.full_like(shap_vals, i), color='grey', alpha=0.5, zorder=1)

    # 绘制 SHAP 蜜蜂群图的散点，添加垂直抖动
    for i in range(len(sorted_feature_names)):
        shap_vals = sorted_shap_values[:, i]
        feature_vals = sorted_features[:, i]
        # 计算归一化颜色
        val_min, val_max = feature_vals.min(), feature_vals.max()
        if val_max == val_min:  # 避免除 0
            norm_vals = np.zeros_like(feature_vals)
        else:
            norm_vals = (feature_vals - val_min) / (val_max - val_min)
        colors = plt.cm.coolwarm(norm_vals)

        # 添加垂直抖动
        y_jitter = i + np.random.uniform(-0.2, 0.2, size=shap_vals.shape)

        # 绘制散点图
        ax_shap.scatter(
            shap_vals,
            y_jitter,
            color=colors,
            alpha=0.6,
            edgecolors='k',
            linewidths=0.5,
            zorder=2,  # 确保散点图在密度图上方
            s=30  # 增大点的大小
        )

    # -----------------------
    # 6) 设置黑色加粗的轴边框和禁用网格
    # -----------------------
    for spine in ax_imp.spines.values():
        spine.set_color('black')
        spine.set_linewidth(1.5)
    for spine in ax_shap.spines.values():
        spine.set_color('black')
        spine.set_linewidth(1.5)
    ax_imp.grid(False)
    ax_shap.grid(False)

    # -----------------------
    # 7) 整体视觉调优
    # -----------------------
    #ax_imp.legend(loc='upper right', fontsize=11)  # 如果不需要图例，可以保持注释状态
    fig.suptitle("SHAP Summary Plot with Feature Importance (Combined)", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # 给标题留空间

    # -----------------------
    # 8) 保存和显示
    # -----------------------
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

# -----------------------------------------------------------------
# 示例：说明如何调用上面的函数
# -----------------------------------------------------------------
def explain_models_with_shap(model_names, X_train, X_test, feature_names):
    """
    对指定的模型进行 SHAP 解释，并生成合并的图表
    """
    save_dir = "E:/jupyter lab/可压性项目产能预测/shap_plots"
    os.makedirs(save_dir, exist_ok=True)

    for model_name in model_names:
        print(f"\n正在为 {model_name} 生成SHAP解释...")

        # 加载保存的模型
        model_path = f'E:/jupyter lab/可压性项目产能预测/models/best_model_{model_name}.joblib'
        pipeline = load(model_path)

        # 数据预处理和特征变换
        if model_name.startswith(('pfs-', 'pfr-')):
            scaler_step = 'ss' if model_name.startswith('pfs-') else 'rs'
            X_scaled = pipeline.named_steps[scaler_step].transform(X_test)
            X_transformed = pipeline.named_steps['poly'].transform(X_scaled)
            degree = pipeline.named_steps['poly'].degree
            transformed_feature_names = []
            for d in range(1, degree + 1):
                for i in range(len(feature_names)):
                    for j in range(i + 1):
                        if i == j:
                            transformed_feature_names.append(f"{feature_names[i]}^{d}")
                        else:
                            transformed_feature_names.append(f"{feature_names[i]}*{feature_names[j]}")
        else:
            scaler_step = 'ss' if model_name.startswith('ps-') else 'rs'
            X_scaled = pipeline.named_steps[scaler_step].transform(X_test)
            X_transformed = pipeline.named_steps['pca'].transform(X_scaled)
            n_components = pipeline.named_steps['pca'].n_components_
            transformed_feature_names = [f"PC{i+1}" for i in range(n_components)]

        # 计算 SHAP 值
        if 'mlp' in pipeline.named_steps:
            background = shap.kmeans(X_transformed, 50)
            mlp_model = pipeline.named_steps['mlp']
            explainer = shap.KernelExplainer(mlp_model.predict, background)
            sample_size = min(100, len(X_test))
            shap_values = explainer.shap_values(X_transformed[:sample_size])
            # 如果是多输出，shap_values 可能是 list，取第一个
            if isinstance(shap_values, list):
                shap_values = shap_values[0]
        elif 'xgb' in pipeline.named_steps:
            xgb_model = pipeline.named_steps['xgb']
            explainer = shap.TreeExplainer(xgb_model)
            shap_values = explainer.shap_values(X_transformed)
            # 对于 XGBoost，shap_values 通常是一个 numpy 数组

        # 绘图并保存
        save_path = os.path.join(save_dir, f'combined_summary_importance_{sanitize_filename(model_name)}.png')
        combined_summary_importance_plot(
            shap_values=shap_values,
            features=X_transformed[:shap_values.shape[0]],  # 修复行数不匹配
            feature_names=transformed_feature_names,
            save_path=save_path
        )

# -----------------------------------------------------------------
# 调用示例
# -----------------------------------------------------------------
# 请确保 X_train 和 X_test 已经定义并且与模型匹配
# 示例：
models_to_explain = ['pfs-nn', 'pfr-nn', 'ps-nn', 'pfs-xgb']
explain_models_with_shap(
    models_to_explain,
    X_train=X_train,  # 替换为你的 X_train 数据
    X_test=X_test,    # 替换为你的 X_test 数据
    feature_names=['DEPTH', 'DT', 'ZDEN', 'GR', 'CNCF', 'M2RX', 'POR', 'PERM', 'SW']
)
